# Plot Consumption

Plot the four-home sample for January-August 2013.

Open this notebook in `notebooks/M5/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

## Read the half-hour sample

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
raw_path = PROJECT_ROOT / "data/pilot/half_hour_sample.csv"
raw = pd.read_csv(raw_path, na_values=["Null", "null"])
raw.head()

## Parse the values and raw times

In [ ]:
raw["tstp"] = pd.to_datetime(raw["tstp"], utc=True)
raw["energy"] = pd.to_numeric(raw["energy(kWh/hh)"])
raw["interval_start"] = raw["tstp"] - pd.Timedelta(minutes=30)
raw.head()

## Keep only the training period

In [ ]:
start = pd.Timestamp("2013-01-01", tz="UTC")
end = pd.Timestamp("2013-09-01", tz="UTC")
train = raw[(raw["interval_start"] >= start) & (raw["interval_start"] < end)].copy()
train.head()

## Use the four development homes

In [ ]:
homes = ["MAC000414", "MAC000675", "MAC001312", "MAC005456"]
sample_home = "MAC000414"
train = train[train["LCLid"].isin(homes)]
train.groupby("LCLid").size()

## List all expected half-hours

In [ ]:
times = pd.date_range(start, end, freq="30min", inclusive="left")
grid = pd.MultiIndex.from_product([homes, times], names=["LCLid", "interval_start"])
len(times), len(grid)

## Place observed energy on the grid

In [ ]:
observed = train.set_index(["LCLid", "interval_start"])["energy"]
energy = observed.reindex(grid)
energy.head()

## Distinguish absent rows from missing measurements

In [ ]:
present = pd.Series(grid.isin(observed.index), index=grid)
missing_rows = ~present
missing_measurements = present & energy.isna()
missing_rows.groupby(level="LCLid").sum()

## Build a simple missing readings table

In [ ]:
coverage = pd.DataFrame(index=pd.Index(homes, name="LCLid"))
coverage["expected_intervals"] = len(times)
coverage["absent_intervals"] = missing_rows.groupby(level="LCLid").sum()
coverage["missing_measurements"] = missing_measurements.groupby(level="LCLid").sum()
coverage["available_intervals"] = energy.notna().groupby(level="LCLid").sum()
coverage["total_missing"] = coverage["absent_intervals"] + coverage["missing_measurements"]
coverage

## Read the example home's curve values

In [ ]:
curve = energy.loc[sample_home]
curve.head()

## Count its available and missing intervals

In [ ]:
curve_available = curve.notna().sum()
curve_missing = curve.isna().sum()
curve_available, curve_missing

## Chart 1: consumption over time

In [ ]:
curve_fig, curve_ax = plt.subplots(figsize=(11, 4))
curve_ax.plot(curve.index, curve, linewidth=0.5, color="#0056b3")
curve_ax.set_title("Example home: half-hour energy, TRAIN Jan-Aug 2013")
curve_ax.set_xlabel("Half-hour start (UTC)")
curve_ax.set_ylabel("Energy (kWh per half-hour)")
curve_fig.tight_layout()
plt.show()

## Keep measured readings for the histogram

In [ ]:
valid_kwh = energy.dropna()
total_missing = energy.isna().sum()
len(valid_kwh), total_missing

## Chart 2: distribution of measured energy

In [ ]:
distribution_fig, distribution_ax = plt.subplots(figsize=(8, 4))
distribution_ax.hist(valid_kwh, bins=40, edgecolor="black", color="#4A90E2")
distribution_ax.set_title("Four-home half-hour energy distribution, TRAIN Jan-Aug 2013")
distribution_ax.set_xlabel("Energy (kWh per half-hour)")
distribution_ax.set_ylabel("Number of measured intervals")
distribution_fig.tight_layout()
plt.show()

## Choose the output folder

In [ ]:
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

## Save the time curve

In [ ]:
curve_path = PROJECT_ROOT / "figures" / "household_consumption_sample.png"
curve_fig.savefig(curve_path, dpi=150)
curve_path

## Save the distribution

In [ ]:
distribution_path = PROJECT_ROOT / "figures" / "consumption_distribution_sample.png"
distribution_fig.savefig(distribution_path, dpi=150)
distribution_path

## Save the missing readings counts

In [ ]:
coverage_path = output_dir / "half_hour_coverage_sample.csv"
coverage.to_csv(coverage_path, index_label="LCLid")
coverage_path

## Prepare the time-curve caption

In [ ]:
curve_caption = (
    f"Home {sample_home}, Jan-Aug 2013. Energy is kWh per half-hour. "
    f"Available: {curve_available}; missing: {curve_missing}. Gaps show missing readings."
)
curve_caption

## Prepare the distribution caption

In [ ]:
distribution_caption = (
    f"Four homes, Jan-Aug 2013. {len(valid_kwh)} half-hour readings in kWh. "
    f"Missing readings excluded: {total_missing}. Measured zeros are included."
)
distribution_caption

## Save the captions

In [ ]:
notes_path = PROJECT_ROOT / "figures" / "consumption_chart_notes_sample.md"
notes = "# M5 sample chart notes\n\n## Time curve\n\n" + curve_caption + "\n\n## Distribution\n\n" + distribution_caption + "\n"
notes_path.write_text(notes, encoding="utf-8")
notes_path